## Check Pytorch Availability

### Import PyTorch
Loads the deep-learning library.

In [2]:
import torch

### Check CUDA availability
Can PyTorch see the GPU? → `True`.

In [3]:
torch.cuda.is_available()

True

### Check PyTorch build & GPU driver
Confirms the CUDA build of PyTorch and that the NVIDIA driver responds.

In [4]:
import torch
import subprocess

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA build: {torch.version.cuda}")

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print("Driver status: OK (nvidia-smi works)") if result.returncode == 0 else print("Driver status: FAILED")
except FileNotFoundError:
    print("Driver status: nvidia-smi NOT FOUND")


PyTorch version: 2.14.0+cu130
PyTorch CUDA build: 13.0
Driver status: OK (nvidia-smi works)


### Show GPU name
Prints which GPU PyTorch will use → `NVIDIA GeForce RTX 4060`.

In [5]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 4060'

### Load datasets

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

### Load, split & scale the dataset
569 patients × 30 features → 80/20 train/test split → each feature scaled to mean 0, std 1.

In [7]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test,y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Inspect scaled features
Peek at the scaled training data → `(455, 30)`, values centred around 0.

In [8]:
X_train_scaled

array([[ 1.69118082,  0.01166163,  1.694079  , ...,  1.5326609 ,
         0.18036408, -0.20793544],
       [ 1.08515471,  0.24434269,  1.07004274, ...,  0.50536464,
         0.52677949, -0.46440693],
       [-0.75805835, -1.04224669, -0.74817201, ..., -0.78536403,
        -1.75102051, -0.57725439],
       ...,
       [-1.52299177, -1.14946247, -1.52254428, ..., -1.73095721,
        -0.51563037, -0.33731106],
       [ 0.2417451 ,  1.30281338,  0.14371358, ..., -1.29555781,
        -2.14014468, -1.67324256],
       [-1.30124766,  0.34699609, -1.29197504, ..., -0.75217215,
        -0.06323398,  0.4395126 ]], shape=(455, 30))

### Inspect labels
Peek at the training labels → 0 = malignant, 1 = benign.

In [9]:
y_train

array([0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1,

### Convert to tensors
NumPy arrays → PyTorch tensors; labels reshaped to `[n, 1]` to match the model output.

In [10]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).float().unsqueeze(1)
y_test_tensor = torch.from_numpy(y_test).float().unsqueeze(1)

### Pair features with labels
Bundles each patient's 30 features with their label.

In [11]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

### Count input features
Number of features per patient (30) → the input size of the first layer.

In [12]:

instanceNo = X_train_scaled_tensor.shape[1]

# result: torch.Size([455, 30])
# meaning there are 30 features per instance

### Batch the training data
Serves shuffled batches of 32 patients to the training loop.

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

### Build Neural Network

In [ ]:
# This is the structure on how to define a class in Pytorch
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()

        # The 2nd param of `nn.Linear` is the output number 
        # for the next layer
        self.fc1 = nn.Linear(instanceNo, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        # Drop out layer is optional
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        # Drop out layer is optional
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x)) # sigmoid is to get value between 0 & 1
        
        return x

### Create model, loss & optimizer
BCE loss = how wrong a prediction is; Adam = how weights get adjusted (lr 0.001).

In [15]:
# Define a loss function and optimizer (learning rate)
# Since it's binary classification, we can choose Binary Cross Entropy Loss
criterion = nn.BCELoss()

# Define a model optimizer (including learning rate)
# Since it's binary classification, we can choose Binary Cross Entropy Loss
model = BCNet()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Train Neural Network

In [16]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss was {running_loss/len(train_loader)}')

Epoch 1: Loss was 0.6458918412526449
Epoch 2: Loss was 0.5186963041623434
Epoch 3: Loss was 0.3571219484011332
Epoch 4: Loss was 0.21806405534346898
Epoch 5: Loss was 0.16411151736974716
Epoch 6: Loss was 0.11647633463144302
Epoch 7: Loss was 0.09984739708403746
Epoch 8: Loss was 0.09591424614191055
Epoch 9: Loss was 0.0808578584343195
Epoch 10: Loss was 0.07976119834929704
Epoch 11: Loss was 0.06987563185393811
Epoch 12: Loss was 0.06713696097334226
Epoch 13: Loss was 0.05512652142594258
Epoch 14: Loss was 0.059663382420937223
Epoch 15: Loss was 0.056271967788537346
Epoch 16: Loss was 0.06748926093180975
Epoch 17: Loss was 0.06334513233353695
Epoch 18: Loss was 0.05842857671280702
Epoch 19: Loss was 0.04741535655533274
Epoch 20: Loss was 0.08004960557445884


### Evaluate test accuracy
Checks predictions on unseen patients → ~97–98% accuracy.

In [17]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()
    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

accuracy

0.9824561476707458

# Phase 2: Export to ONNX

### Prepare model for export
`eval()` switches Dropout off (deterministic, as a circuit needs); CPU for a device-independent export.

In [18]:
import os

os.makedirs("artifacts", exist_ok=True)   # folder for all ZK files

model.eval()        # inference mode: Dropout switches OFF
model.to("cpu")     # export from CPU so the result doesn't depend on the GPU


BCNet(
  (fc1): Linear(in_features=30, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

### Pick the sample patient
Test patient #0, shape `[1, 30]`: the input we export with and later prove.

In [19]:
sample_input = X_test_scaled_tensor[0:1]   # first test patient; [0:1] keeps the batch dimension
sample_input.shape                         # torch.Size([1, 30])

torch.Size([1, 30])

### Export to ONNX
Records the model's operations + weights into `artifacts/network.onnx` (opset 18 for EZKL).

In [20]:
onnx_program = torch.onnx.export(
    model,
    (sample_input,),                 # example inputs go in a tuple
    input_names=["input"],
    output_names=["output"],
    opset_version=18,               # the newest version EZKL is tested with
    dynamo=True,                    # the official exporter (the default; written out to be clear)
)

onnx_program.save("artifacts/network.onnx")

W0927 01:04:51.809000 22264 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W0927 01:04:51.811000 22264 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W0927 01:04:51.812000 22264 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W0927 01:04:51.812000 22264 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d
W0927 01:04:51.893000 22264 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `BCNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BCNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


### Check the ONNX graph
Validates the file and lists its ops → `Gemm, Relu, Gemm, Relu, Gemm, Sigmoid`.

In [21]:
import onnx

graph = onnx.load("artifacts/network.onnx")
onnx.checker.check_model(graph)

print("Operations:", [node.op_type for node in graph.graph.node])
print("Opset:", graph.opset_import[0].version)
print("File size:", os.path.getsize("artifacts/network.onnx"), "bytes")


Operations: ['Gemm', 'Relu', 'Gemm', 'Relu', 'Gemm', 'Sigmoid']
Opset: 18
File size: 22922 bytes


### Save `input.json`
Writes the sample patient's 30 values in EZKL's input format.

In [22]:
# Save `input.json` for EZKL inference
import json

input_data = {"input_data": [sample_input[0].tolist()]}

with open("artifacts/input.json", "w") as f:
    json.dump(input_data, f)

print("Features:", len(input_data["input_data"][0]))
print("First 3 values:", input_data["input_data"][0][:3])


Features: 30
First 3 values: [0.26129433512687683, 0.07781604677438736, 0.3365975320339203]


### Record PyTorch reference prediction
The float answer to compare against the circuit's integer answer later.

In [23]:
# Record what PyTorch predicts for this patient
with torch.no_grad():
    pytorch_output = model(sample_input).item()

print("PyTorch prediction:", pytorch_output)
print("True label:", y_test[0])


PyTorch prediction: 0.0004888330586254597
True label: 0


# Phase 3: EZKL local proof

### Generate default settings
Visibility: inputs/outputs public, weights fixed → default `settings.json` (scale 7, logrows 17).

In [24]:
# Generate the default settings

import ezkl

MODEL    = "artifacts/network.onnx"
INPUT    = "artifacts/input.json"
CALIB    = "artifacts/calibration.json"
SETTINGS = "artifacts/settings.json"

run_args = ezkl.PyRunArgs()
run_args.input_visibility  = "public"   # verifier sees the patient's 30 features
run_args.output_visibility = "public"   # verifier sees the prediction
run_args.param_visibility  = "fixed"    # weights built into the circuit: hidden but bound

ezkl.gen_settings(MODEL, SETTINGS, py_run_args=run_args)

s = json.load(open(SETTINGS))["run_args"]
print("Defaults -> scale:", s["input_scale"], "| logrows:", s["logrows"], "| lookup_range:", s["lookup_range"])


Defaults -> scale: 7 | logrows: 17 | lookup_range: [-32768, 32768]


### Build calibration data
50 real training patients → `calibration.json`, so EZKL sees the real value range.

In [25]:
# Build the calibration data for the circuit

calib_data = {"input_data": [X_train_scaled[:50].flatten().tolist()]}

with open(CALIB, "w") as f:
    json.dump(calib_data, f)

print("Values in calibration file:", len(calib_data["input_data"][0]))   # 50 × 30 = 1500


Values in calibration file: 1500


### Calibrate scale & logrows
Picks the precision (scale 11 = 1/2048 steps) and circuit size (logrows 18) from real data. Red error lines are normal.

In [26]:
# Calibrate the circuit settings based on the calibration data

ok = ezkl.calibrate_settings(CALIB, MODEL, SETTINGS, "resources")
print("Calibration OK:", ok)

s = json.load(open(SETTINGS))["run_args"]
print("Calibrated -> scale:", s["input_scale"], "| logrows:", s["logrows"], "| lookup_range:", s["lookup_range"])


[tensor] decomposition error: integer -25307181295 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 192060900446 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 768218271481 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 286719935006 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 1146906231969 is too large to be represented by base 16384 and n 2
forward pass failed: "failed to forward: [halo2] General synthesis error"
[tensor] decomposition error: integer 3072935751545 is too large to be represented by base 16384 and n 2
forw

Calibration OK: True
Calibrated -> scale: 11 | logrows: 18 | lookup_range: [-125152, 57106]


### Compile the circuit
ONNX + settings → `network.compiled` (the circuit, weights baked in).

In [27]:
# Compile
COMPILED = "artifacts/network.compiled"

ezkl.compile_circuit(MODEL, COMPILED, SETTINGS)
print("Compiled:", os.path.getsize(COMPILED), "bytes")


Compiled: 135316 bytes


### Resume cell (after kernel restart)
Re-creates imports and paths without retraining; sets `HOME`, which EZKL needs on Windows.

In [2]:
# Reread the settings to get the logrows for the SRS. Useful when the kernel is restarted and the SRS is already cached.
import os, sys, json, subprocess, ezkl
os.environ.setdefault("HOME", os.path.expanduser("~"))   # Windows has no HOME; EZKL needs it

MODEL, INPUT, CALIB, SETTINGS = ("artifacts/network.onnx", "artifacts/input.json",
                                 "artifacts/calibration.json", "artifacts/settings.json")
COMPILED = "artifacts/network.compiled"

### Get the SRS
Public KZG setup file (like circom's `.ptau`), sized to logrows; downloaded only if missing.

In [3]:
# Get the SRS: download only if it's not cached yet
import sys, subprocess

logrows = json.load(open(SETTINGS))["run_args"]["logrows"]
SRS = os.path.expanduser(f"~/.ezkl/srs/kzg{logrows}.srs")

if not os.path.exists(SRS):
    print(f"kzg{logrows}.srs not found, downloading...")
    result = subprocess.run(
        [sys.executable, "-c",
         f"import asyncio, ezkl; print(asyncio.run(ezkl.get_srs({SETTINGS!r})))"],
        capture_output=True, text=True,
    )
    print(result.stdout, result.stderr[-500:])

print("SRS:", SRS, os.path.getsize(SRS), "bytes")


SRS: C:\Users\rasyi/.ezkl/srs/kzg18.srs 33554692 bytes


### Generate proving & verifying keys
Circuit + SRS → `pk.key` (1.3 GB, prover) and `vk.key` (0.6 MB, verifier; commits to the weights).

In [4]:
# Generate the keys
VK = "artifacts/vk.key"
PK = "artifacts/pk.key"
ezkl.setup(COMPILED, VK, PK, srs_path=SRS)

print("Verifying key:", os.path.getsize(VK), "bytes")
print("Proving key:  ", os.path.getsize(PK), "bytes")


Verifying key: 591367 bytes
Proving key:   1309214827 bytes


### Generate the witness
Runs the patient through the circuit in integer math → output `1/2048` ≈ PyTorch's 0.000489.

In [5]:
# Generate the witness
WITNESS = "artifacts/witness.json"

ezkl.gen_witness(INPUT, COMPILED, WITNESS)

w = json.load(open(WITNESS))
print("Circuit output:", w["pretty_elements"]["rescaled_outputs"])
print("Raw field element:", w["outputs"])


Circuit output: [['0.00048828125']]
Raw field element: [['0100000000000000000000000000000000000000000000000000000000000000']]


### Generate the proof
Witness + proving key → `proof.json` (~25 KB, ~9 s).

In [6]:
# Generate the proof
PROOF = "artifacts/proof.json"

ezkl.prove(WITNESS, COMPILED, PK, PROOF, srs_path=SRS)
print("Proof size:", os.path.getsize(PROOF), "bytes")


Proof size: 25001 bytes


### Verify the proof locally
Checks the proof with the verifying key → `True` (tampered → `False`).

In [7]:
# Verify the proof locally
try:
    ok = ezkl.verify(PROOF, SETTINGS, VK, srs_path=SRS)
except RuntimeError as e:
    ok = False
    print("Verification failed:", e)

print("Proof valid:", ok)


Proof valid: True


# Phase 4: Verify on Chain

### Generate the Solidity verifier
Turns `vk.key` into `Verifier.sol` (`Halo2Verifier.verifyProof(bytes, uint256[])`).

In [8]:
SOL = "artifacts/Verifier.sol"
ABI = "artifacts/Verifier.abi"

await ezkl.create_evm_verifier(VK, SETTINGS, SOL, ABI, srs_path=SRS)

print("Verifier.sol:", os.path.getsize(SOL), "bytes")
print(open(ABI).read())


Verifier.sol: 81096 bytes
[{"type":"function","name":"verifyProof","inputs":[{"name":"proof","type":"bytes","internalType":"bytes"},{"name":"instances","type":"uint256[]","internalType":"uint256[]"}],"outputs":[{"name":"","type":"bool","internalType":"bool"}],"stateMutability":"nonpayable"}]


### Foundry setup
Paths, local RPC, Anvil's public dev key, and a `run()` helper that calls forge/anvil/cast.

In [9]:
# Deploy the verifier contract to a local Ethereum node (Anvil)
import shutil, time

FOUNDRY = os.path.expanduser("~/.foundry/bin")
RPC = "http://127.0.0.1:8545"
KEY = "0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80"   # Anvil's public dev account #0

def run(*cmd, cwd=None):
    r = subprocess.run([os.path.join(FOUNDRY, cmd[0]), *cmd[1:]], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stderr.strip() or r.stdout.strip())
    return r.stdout.strip()


### Build the verifier contract
Copies the latest `Verifier.sol` into `contracts/` and compiles it → 15,194-byte runtime (< 24 KB limit).

In [ ]:
# Build the verifier contract with Foundry
shutil.copy(SOL, "contracts/src/Verifier.sol")
print(run("forge", "build", "--sizes", cwd="contracts"))


No files changed, compilation skipped

â•­---------------+------------------+-------------------+--------------------+---------------------â•®
| Contract      | Runtime Size (B) | Initcode Size (B) | Runtime Margin (B) | Initcode Margin (B) |
+=================================================================================================+
| Halo2Verifier | 15,194           | 15,223            | 9,382              | 33,929              |
â•°---------------+------------------+-------------------+--------------------+---------------------â•¯


### Start local chain (Anvil)
Runs a local Ethereum node in the background → block 0.

In [11]:
# Start Anvil (local chain) in the background
anvil = subprocess.Popen([os.path.join(FOUNDRY, "anvil"), "--port", "8545"],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("Block number:", run("cast", "block-number", "--rpc-url", RPC))


Block number: 0


### Deploy the verifier
Deploys `Halo2Verifier` to Anvil → contract address.

In [ ]:
# Deploy the verifier contract to Anvil
out = run("forge", "create", "src/Verifier.sol:Halo2Verifier",
          "--rpc-url", RPC, "--private-key", KEY, "--broadcast", "--json", cwd="contracts")
VERIFIER_ADDR = json.loads(out)["deployedTo"]
print("Verifier deployed at:", VERIFIER_ADDR)



Verifier deployed at: 0x5FbDB2315678afecb367f032d93F642f64180aa3


### Verify the proof on-chain
Packs proof + 31 public values into calldata and calls `verifyProof` → `True`, ~639k gas.

In [13]:
# Verify the proof on-chain
CALLDATA = "artifacts/calldata.bytes"
calldata = ezkl.encode_evm_calldata(PROOF, CALLDATA)
hexdata = "0x" + calldata.hex()

result = run("cast", "call", VERIFIER_ADDR, "--data", hexdata, "--rpc-url", RPC)
print("verifyProof ->", int(result, 16) == 1)
print("Gas to verify:", run("cast", "estimate", VERIFIER_ADDR, hexdata, "--rpc-url", RPC))


verifyProof -> True
Gas to verify: 639011


### On-chain tamper test
Flips the output value in the calldata → the verifier reverts → `False`.

In [14]:
# On-chain tamper test 

# Find where the output value sits in the calldata
inst_offset = int.from_bytes(calldata[36:68], "big")                       # where the instances array starts
n_inst      = int.from_bytes(calldata[4+inst_offset:4+inst_offset+32], "big")  # 31 = 30 inputs + 1 output
out_byte    = 4 + inst_offset + 32 + 32*n_inst - 1                         # last byte of the output value

bad = bytearray(calldata)
bad[out_byte] ^= 1                                                         # output 1 → 0

try:
    r = run("cast", "call", VERIFIER_ADDR, "--data", "0x" + bad.hex(), "--rpc-url", RPC)
    ok = int(r, 16) == 1
except RuntimeError as e:
    ok = False
    print("Rejected:", str(e).splitlines()[0])
print("Tampered proof valid:", ok)


Rejected: Error: server returned an error response: error code 3: execution reverted, data: "0x"
Tampered proof valid: False


### Stop the chain
Shuts down Anvil and frees port 8545.

In [15]:
# Stop Anvil
anvil.terminate()
print("Anvil stopped")

Anvil stopped
